# Exploratory Data Analysis — Customer Churn Prediction

I built this notebook to explore the IBM Telco Customer Churn dataset before building classification models. My goal is to understand what customer attributes are associated with churn, clean up any missing or anomalous values, and check for practical issues like feature collinearity.

**Notebook Outline**:
1. Project objective and dataset loading
2. Initial data inspection
3. Cleaning and handling missing values
4. Exploratory analysis and visualizations
5. Observations from the data
6. Feature relationships and potential issues
7. Questions to investigate during modelling


## 1. Project Objective and Dataset Loading
We want to predict whether a customer will cancel their telecom service (`Churn` = Yes/No). Let's load the required libraries and the raw CSV file.

In [1]:
import sys
from pathlib import Path

# Add project src directory to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

from churn_prediction.config import get_config
from churn_prediction.data_loader import load_raw_data, clean_raw_data, validate_raw_data

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('Environment ready.')

Environment ready.


In [2]:
config = get_config()
raw_df = load_raw_data(config.raw_data_path)
print(f'Rows: {raw_df.shape[0]}, Columns: {raw_df.shape[1]}')
raw_df.head(5)

Rows: 7043, Columns: 21


## 2. Initial Data Inspection
Let's check the schema, data types, and verify that customerID has no duplicate records.

In [3]:
print('Duplicate customer IDs:', raw_df['customerID'].duplicated().sum())
print('\nColumn data types:')
print(raw_df.dtypes)

Duplicate customer IDs: 0

Column data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


## 3. Cleaning and Handling Missing Values
Notice that `TotalCharges` is stored as an `object` (string) rather than a float. Let's see why it wasn't parsed as a number.

In [4]:
# Find rows where TotalCharges contains blank whitespace
blank_tc = raw_df[raw_df['TotalCharges'].astype(str).str.strip() == '']
print(f'Rows with blank TotalCharges: {len(blank_tc)}')
blank_tc[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'Churn']]

Rows with blank TotalCharges: 11


**Observation**: All 11 customers with blank `TotalCharges` have `tenure == 0`. They just joined the service during the current billing cycle and haven't completed a month yet. None of them have churned (`Churn == No`).

**Decision**: Setting `TotalCharges = 0.0` for `tenure == 0` is logically consistent with how billing cycles work. Let's clean the dataset and confirm no missing values remain.

In [5]:
clean_df = clean_raw_data(raw_df, config)
print('Missing values after cleaning:')
print(clean_df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].isna().sum())
print('\nTarget distribution:')
print(clean_df['Churn'].value_counts(normalize=True).round(4) * 100)

Missing values after cleaning:
tenure            0
MonthlyCharges    0
TotalCharges      0
Churn             0
dtype: int64

Target distribution:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


## 4. Exploratory Analysis and Visualizations
Let's look at the class balance and how churn varies across numerical and categorical features.

In [6]:
churn_counts = clean_df['Churn'].value_counts()
churn_pct = (churn_counts / len(clean_df)) * 100
labels = ['Retained (0)', 'Churned (1)']

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=labels, y=churn_counts.values, hue=labels, palette=['#2ca02c', '#d62728'], legend=False, ax=ax)
for i, p in enumerate(ax.patches):
    ax.annotate(f'{int(p.get_height()):,} ({churn_pct.iloc[i]:.1f}%)',
                (p.get_x() + p.get_width() / 2., p.get_height() / 2.),
                ha='center', va='center', fontsize=12, color='white', weight='bold')
plt.title('Target Class Distribution', fontsize=13, weight='bold')
plt.ylabel('Customers')
plt.show()

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
clean_df['Churn_Label'] = clean_df['Churn'].map({1: 'Churned', 0: 'Retained'})
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
titles = ['Tenure (Months)', 'Monthly Charges ($)', 'Total Charges ($)']

for ax, col, title in zip(axes, num_cols, titles):
    sns.kdeplot(data=clean_df, x=col, hue='Churn_Label', common_norm=False, fill=True, alpha=0.35, ax=ax, palette={'Retained': '#2ca02c', 'Churned': '#d62728'})
    ax.set_title(title, fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

In [8]:
cat_cols = ['Contract', 'PaymentMethod', 'InternetService', 'TechSupport']
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, col in zip(axes.flatten(), cat_cols):
    rate_df = clean_df.groupby(col)['Churn'].mean().reset_index()
    rate_df['Churn_Rate'] = rate_df['Churn'] * 100
    rate_df = rate_df.sort_values(by='Churn_Rate', ascending=False)
    sns.barplot(data=rate_df, x=col, y='Churn_Rate', hue=col, palette='Blues_r', legend=False, ax=ax)
    ax.set_title(f'Churn Rate by {col}', fontsize=12, weight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_xlabel('')
    ax.set_ylim(0, 60)
    ax.tick_params(axis='x', rotation=15)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1f}%',
                    (p.get_x() + p.get_width() / 2., p.get_height() + 1.0),
                    ha='center', fontsize=10, weight='bold')
plt.tight_layout()
plt.show()

In [9]:
# Querying the SQLite database to confirm core breakdowns
conn = sqlite3.connect(str(config.database_path))
sql_df = pd.read_sql_query('''
SELECT 
    contract,
    internet_service,
    COUNT(*) AS total_customers,
    SUM(churn_numeric) AS churned_customers,
    ROUND(AVG(churn_numeric) * 100.0, 2) AS churn_rate_pct,
    ROUND(AVG(monthly_charges), 2) AS avg_monthly_bill
FROM telco_churn
GROUP BY contract, internet_service
ORDER BY churn_rate_pct DESC;
''', conn)
conn.close()
sql_df

## 5. Observations from the Data
Here are the clearest patterns from the exploratory visualizations:
- **Class Imbalance**: Churn occurs in 26.5% of accounts (1,869 out of 7,043). A naive model predicting 'never churn' would be 73.5% accurate but useless. Accuracy alone is not an appropriate evaluation metric.
- **Tenure**: Churn is heavily concentrated in early customer life. Customers who make it past 24–36 months are far less likely to leave.
- **Contract Type**: Strongest categorical separator. Month-to-month contracts have a **42.7%** churn rate, compared to **11.3%** for one-year and **2.8%** for two-year contracts.
- **Payment Method**: Electronic check users have a **45.3%** churn rate, compared to ~15–19% for automated methods (Bank transfer or Credit card).
- **Internet Service & Tech Support**: Fiber optic subscribers churn at **41.9%**, while DSL is **19.0%**. Customers without Tech Support churn at **41.6%**, compared to **15.2%** for those with it.

## 6. Feature Relationships and Potential Issues
Let's examine correlation among numerical features. In particular, `TotalCharges` is conceptually close to `tenure * MonthlyCharges`.

In [10]:
# Check correlation between TotalCharges and tenure * MonthlyCharges
approx_total = clean_df['tenure'] * clean_df['MonthlyCharges']
actual_total = clean_df['TotalCharges']
collinearity_corr = approx_total.corr(actual_total)
print(f'Pearson correlation between TotalCharges and (tenure * MonthlyCharges): {collinearity_corr:.5f}')

plt.figure(figsize=(6, 4.5))
corr = clean_df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.3f', linewidths=0.5)
plt.title('Numerical Correlation Heatmap', fontsize=12, weight='bold')
plt.show()

Pearson correlation between TotalCharges and (tenure * MonthlyCharges): 0.99956


**Observation**: `TotalCharges` has a **0.9996** correlation with `tenure * MonthlyCharges`. 
In linear models (like Logistic Regression), including both `tenure`, `MonthlyCharges`, and `TotalCharges` introduces severe multicollinearity, causing standard errors to inflate and coefficients to act as suppressors. 
We should test whether dropping `TotalCharges` during model training impacts cross-validation scores.

## 7. Questions to Investigate During Modelling
Based on this exploration, here are the main questions I plan to test during modeling:
1. **Baseline vs. Ensembles**: Does Random Forest or Gradient Boosting meaningfully outperform Logistic Regression on this dataset, or is linear classification sufficient?
2. **Feature Ablation**: Does dropping `TotalCharges` hurt validation PR-AUC, or does it simplify the model without performance loss?
3. **Threshold Selection**: If the business goal is to reach out to at-risk customers, what classification threshold balances recall (catching churners) against precision (limiting false alarms and wasted retention offers)?
4. **Leakage Prevention**: Ensure preprocessing scalers and one-hot encoders are strictly fit on training folds only.